In [ ]:
# Display the output of all lines in a cell
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Creating a lightweight (image-free) version of LIBERO dataset in LeRobot format

HuggingFace's DataStudio viewer and SQL tool do now work well for large datasets. Hence, in order to explore the contents of the LIBERO dataset in LeRobot format (which can be downloaded from https://huggingface.co/datasets/physical-intelligence/libero), we can download it and create a "lightweight" version of it without the images, which can be easily stored as a CSV file (of size ~100M). This version can be the opened as e.g. pandas DataFrame.

In what follows we assume that we had already downloaded the dataset to a local path (which can also be mounted from an Azure storage account, for example, in order to avoid running out of space if we're working on AzureML VM, for example), which will be passed to the method LeRobotDataset below to avoid it downloading the dataset again from HF. 

In [ ]:
from pprint import pprint
import torch
import pandas as pd
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

In [ ]:
# Load libero
repo_id = "physical-intelligence/libero"

# We can have a look and fetch its metadata to know more about it:
ds_meta = LeRobotDatasetMetadata(repo_id)

In [ ]:
# Print some metadata information about the dataset
print(f"Total number of episodes: {ds_meta.total_episodes}")
print(f"Average number of frames per episode: {ds_meta.total_frames / ds_meta.total_episodes:.3f}")
print(f"Frames per second used during data collection: {ds_meta.fps}")
print(f"Robot type: {ds_meta.robot_type}")
print(f"keys to access images from cameras: {ds_meta.camera_keys=}\n")

print("Tasks:")
print(ds_meta.tasks)
print("Features:")
pprint(ds_meta.features)

# You can also get a short summary by simply printing the object:
print(ds_meta)

In [ ]:
# Load dataset from local path (in our case, we chose to mount an Azure storage account on an AzureML VM using blobfuse2, and then mounted it on the devcontainer on the /mnt/data folder). For completeness, we have also added a sample blobfuse2 config file (blobfuse_config.yaml).
dataset = LeRobotDataset(repo_id, download_videos=False, root = "/mnt/data/physical-intelligence/libero")

In [ ]:
# The most efficient way to iterate over the dataset is to use a pytorch DataLoader. We will use it to creat a DataFrame with all the data except images.
# The dataset is large, so this will take a few minutes to run.
dataloader = torch.utils.data.DataLoader(
    dataset,
    num_workers=0, # Adjust based on your system's capabilities
    batch_size=64,
    shuffle=False,
)

all_rows = []
batch_number = 0
for batch in dataloader:
    # Convert tensors to numpy or lists if needed
    batch_dict = {}
    for key, value in batch.items():
        if key not in ["image", "wrist_image"]:
            if hasattr(value, 'numpy'):
                batch_dict[key] = value.numpy().tolist()
            else:
                batch_dict[key] = value  # e.g., strings or already lists
    batch_number += 1    
    # print every 100 batches to monitor progress
    if batch_number % 100 == 0:
        print(f"Processed batch {batch_number}")


    # Transpose the batch_dict to row-wise format
    batch_rows = [dict(zip(batch_dict, t)) for t in zip(*batch_dict.values())]
    all_rows.extend(batch_rows)

# Create a DataFrame from the collected rows
df = pd.DataFrame(all_rows)

In [ ]:
# Save to CSV
df.to_csv("data/libero_dataset_no_images.csv")